# 6장 (실습 2): Hybrid 경계에 가명처리 검문소 두기

> 책 『AI 보안, 클라우드에서 시작한다』 6장 [실습] 2번을 노트북에서 단계별로 실행하는 버전입니다. 6-3 "Hybrid AI의 데이터 이동 통제"의 흐름(가명처리 → 외부 요약 → 사내 복원)을 작은 Python 프로그램으로 재현합니다. 소요 시간은 30분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 검문소 없이 보냈을 때 **외부 AI 서비스가 실제로 받는 내용**을 직접 확인합니다.
2. 가명처리 → 외부 요약 → 사내 복원 흐름에서 **가명·실명 대응표가 어디에 남는지** 이해합니다.
3. 규칙(basic, strict)마다 **경계를 넘은 정보**를 자동으로 대조해, 규칙만으로 막지 못하는 정보를 찾습니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | 예정 |
| 1 | 실습 파일 살펴보기 | 예정 |
| 2 | 검문소 없이 보내기 | 예정 |
| 3 | 검문소를 거쳐 보내기 (basic) | 예정 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | 예정 |
| 5 | 규칙만으로 부족할 때 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

## 실습 시나리오

온라인 쇼핑몰 "한빛몰"은 상담 기록을 외부 AI 서비스로 요약하려 합니다. 상담 원문은 사내에 두고, 이름과 전화번호를 가명으로 바꾼 내용만 밖으로 보낸 뒤, 돌아온 요약을 사내에서 실명과 다시 연결하는 구조입니다. 6-3에서 본 것처럼 핵심 질문은 하나입니다. **"실제로 무엇이 경계를 넘는가?"**

```text
 사내 원문 ─▶ [검문소: 가명처리] ─▶ 외부 AI(fake_ai.py) ─▶ 가명 요약 ─▶ [검문소: 사내 복원] ─▶ 사용자
                   │                       │
              대응표(사내에만)        received.log (외부가 받은 것)
```

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Python 3만 있으면 되고 외부 패키지와 Docker는 쓰지 않습니다. 외부 AI 서비스는 받은 내용을 그대로 기록하는 흉내 서버(`hybrid/fake_ai.py`)로 대신하므로 API Key가 필요 없습니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch06-aiaas-vendor-evaluation` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch06-aiaas-vendor-evaluation && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch06-aiaas-vendor-evaluation` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 23일 Ubuntu 24.04(WSL2), Python 3.12에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch06_lab2_pseudonymization_checkpoint_executed.ipynb`입니다.

## 0. 실습 준비: 흉내 AI 서비스 띄우기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 실습 파일 살펴보기 | 예정 |
| 2 | 검문소 없이 보내기 | 예정 |
| 3 | 검문소를 거쳐 보내기 (basic) | 예정 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | 예정 |
| 5 | 규칙만으로 부족할 때 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

실습 파일을 홈 디렉터리의 `~/hybrid-lab`으로 복사하고, 외부 AI 서비스 흉내(`fake_ai.py`)를 노트북 뒤에서 실행합니다. 터미널에서 `python3 fake_ai.py &`로 띄우는 것과 같습니다. 흉내 서버는 받은 글을 `~/hybrid-lab/received.log`에 남깁니다.

In [ ]:
import shutil
import socket
import sys
import time

WORK = Path.home() / "hybrid-lab"
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(LAB / "hybrid", WORK)
RECEIVED = WORK / "received.log"

fake_ai = subprocess.Popen([sys.executable, "fake_ai.py"], cwd=WORK,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(50):                                   # 8081 포트가 열릴 때까지 기다린다
    try:
        socket.create_connection(("127.0.0.1", 8081), timeout=0.2).close()
        break
    except OSError:
        time.sleep(0.2)
print(f"외부 AI 서비스(흉내) 실행 중: PID {fake_ai.pid}, http://127.0.0.1:8081")

`gateway.py`를 실행하는 함수를 만듭니다. 매번 `received.log`를 비우고 실행해, **이번 실행에서 경계를 넘은 내용만** 남게 합니다.

In [ ]:
def run_gateway(*args):
    """received.log를 비우고 gateway.py를 실행해 (화면 출력, 외부가 받은 내용)을 돌려줍니다."""
    RECEIVED.write_text("")
    out = subprocess.run([sys.executable, "gateway.py", "records.txt", *args], cwd=WORK,
                         capture_output=True, text=True).stdout
    return out, RECEIVED.read_text()


RESULTS = {}   # 단계별로 외부가 받은 내용을 모아 4단계에서 비교합니다

## 1. 실습 파일 살펴보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | ✅ 완료 |
| 1 | 실습 파일 살펴보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 검문소 없이 보내기 | 예정 |
| 3 | 검문소를 거쳐 보내기 (basic) | 예정 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | 예정 |
| 5 | 규칙만으로 부족할 때 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

| 파일 | 역할 |
|---|---|
| `records.txt` | 상담 기록 3건. 이름, 전화번호와 함께 상담 내용 속에 계좌번호, 주소, 띄어 쓴 전화번호가 섞여 있음 |
| `customers.csv` | 사내 고객 DB를 흉내 낸 이름·전화번호 목록 |
| `fake_ai.py` | 외부 AI 서비스 흉내. 받은 내용을 `received.log`에 남기고 간단한 요약을 돌려줌 |
| `gateway.py` | 경계의 검문소. 가명처리 → 외부 호출 → 사내 복원 |

In [ ]:
for f in ["records.txt", "customers.csv"]:
    print(f"── {f}")
    print((WORK / f).read_text())

상담 기록에서 **외부로 나가면 안 되는 정보**를 미리 목록으로 만들어 둡니다. 2~4단계마다 이 목록이 `received.log`에 남았는지 자동으로 대조합니다.

In [ ]:
SENSITIVE = {
    "고객 이름": ["김민준", "이서연", "박지호"],
    "표준 형식 전화번호": ["010-2481-7730", "010-5539-1204", "010-7712-3368"],
    "띄어 쓴 전화번호": ["010 7712 3368"],
    "계좌번호": ["123456-01-789012"],
    "주소": ["서울시 마포구 월드컵로 15"],
}


def leaked(received):
    """외부가 받은 내용에 남은 민감정보를 항목별로 셉니다."""
    return {kind: sum(v in received for v in values) for kind, values in SENSITIVE.items()}


print("검사할 민감정보:", sum(len(v) for v in SENSITIVE.values()), "개")

## 2. 검문소 없이 보내기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | ✅ 완료 |
| 1 | 실습 파일 살펴보기 | ✅ 완료 |
| 2 | 검문소 없이 보내기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 검문소를 거쳐 보내기 (basic) | 예정 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | 예정 |
| 5 | 규칙만으로 부족할 때 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

`--direct`는 검문소 없이 원문을 그대로 보내는 비교용 옵션입니다.

In [ ]:
out, RESULTS["검문소 없음"] = run_gateway("--direct")
print("── 외부 서비스가 실제로 받은 내용 (received.log)")
print(RESULTS["검문소 없음"])
print("── 남은 민감정보:", leaked(RESULTS["검문소 없음"]))

이름, 전화번호, 계좌번호, 주소가 모두 외부로 나갔습니다. 직원이 공개형 AI에 상담 기록을 붙여 넣을 때(4-2)와 같은 상태입니다.

> 🙋 (생각해 보기)
> 실제 외부 AI 서비스에서는 공급자 쪽에 무엇이 저장되는지 우리가 직접 볼 수 없습니다. 이 실습의 `received.log`는 그 "보이지 않는 쪽"을 보여 주는 장치입니다. 실무에서는 같은 기록을 **우리 쪽 검문소에서** 남겨야 합니다(5단계).

## 3. 검문소를 거쳐 보내기 (basic)

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | ✅ 완료 |
| 1 | 실습 파일 살펴보기 | ✅ 완료 |
| 2 | 검문소 없이 보내기 | ✅ 완료 |
| 3 | 검문소를 거쳐 보내기 (basic) | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 규칙을 강화해 다시 보내기 (strict) | 예정 |
| 5 | 규칙만으로 부족할 때 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 3.1. basic 규칙으로 보내기

basic 규칙은 고객 DB에 있는 이름과 표준 형식 전화번호만 가명으로 바꿉니다.

In [ ]:
out, RESULTS["basic 규칙"] = run_gateway("--rules", "basic")
print(out)

### 3.2. 대응표는 어디에 있나

가명(`고객01`)과 실명(`김민준`)을 잇는 대응표는 `gateway.py`의 `pseudonymize()`가 만들어 **사내 검문소의 메모리에만** 둡니다. 함수를 직접 불러 대응표를 꺼내 봅니다.

In [ ]:
sys.path.insert(0, str(WORK))
os.chdir(WORK)                       # gateway.py가 customers.csv를 현재 폴더에서 읽는다
import gateway

original = (WORK / "records.txt").read_text().strip()
masked, mapping = gateway.pseudonymize(original, gateway.load_customers(), "basic")
os.chdir(LAB)
table(["가명", "원래 값"], mapping.items())
print("대응표의 원래 값이 외부로 나간 내용에 있나?", any(v in RESULTS["basic 규칙"] for v in mapping.values()))

고객 DB에 있는 이름과 전화번호는 가명으로 바뀌어 나갔고, 돌아온 요약은 사내에서 실명으로 복원되었습니다. 가명과 실명의 대응표는 사내 검문소 안에만 있습니다.

### 3.3. 그런데 무엇이 남았나

In [ ]:
print("── 외부가 받은 내용")
print(RESULTS["basic 규칙"])
print("── 남은 민감정보:", leaked(RESULTS["basic 규칙"]))

외부로 보낸 내용을 다시 보면 **계좌번호, 주소, 띄어 쓴 전화번호**가 그대로 남아 있습니다. 고객이 상담 중에 직접 말한 정보라 고객 DB에 없고, 정해진 형식도 아니기 때문입니다. 6-3에서 "상담 내용 안에 고객이 직접 말한 계좌번호나 주소가 섞여 있어 가명처리를 피해 간다"고 한 바로 그 경우입니다.

## 4. 규칙을 강화해 다시 보내기 (strict)

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | ✅ 완료 |
| 1 | 실습 파일 살펴보기 | ✅ 완료 |
| 2 | 검문소 없이 보내기 | ✅ 완료 |
| 3 | 검문소를 거쳐 보내기 (basic) | ✅ 완료 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 규칙만으로 부족할 때 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

`strict` 규칙은 숫자 모양(정규식)으로 계좌번호와 띄어 쓴 전화번호를 찾아 가립니다. `gateway.py`에서 해당 부분을 봅니다.

> 🔎 (용어 풀이) **정규식(정규 표현식, regex)**: 글자의 "모양"을 적는 약속입니다. 예를 들어 `\d{3}-\d{4}-\d{4}`는 "숫자 3개, 하이픈, 숫자 4개, 하이픈, 숫자 4개"라는 뜻이라 `010-1234-5678` 같은 전화번호를 찾습니다. 모양이 정해진 정보(전화번호, 계좌번호)는 잘 찾지만, 주소처럼 모양이 일정하지 않은 정보는 찾기 어렵습니다.

In [ ]:
src = (WORK / "gateway.py").read_text()
print(src[src.index('    if rules == "strict":'):src.index("    return text, table")])

In [ ]:
out, RESULTS["strict 규칙"] = run_gateway("--rules", "strict")
print("── 외부가 받은 내용")
print(RESULTS["strict 규칙"])
print("── 남은 민감정보:", leaked(RESULTS["strict 규칙"]))

### 4.1. 세 번의 결과 비교

세 번 실행에서 외부가 받은 내용을 대조해 표로 만듭니다.

In [ ]:
def cell(kind, received):
    n, total = leaked(received)[kind], len(SENSITIVE[kind])
    return "가림" if n == 0 else ("**나감**" if n == total else f"일부 나감 {n}/{total}")


table(["정보", *RESULTS.keys()], [(k, *(cell(k, r) for r in RESULTS.values())) for k in SENSITIVE])

`[표 실습6-2]` 규칙에 따라 경계를 넘은 정보 — 위 표가 실행 결과로 만든 표입니다. 책의 표에서 "가명"과 "가림"은 모두 위 표의 "가림"에 해당합니다.

strict 규칙에서도 **주소**는 여전히 나갑니다. 주소는 정해진 숫자 형식이 없어 간단한 규칙으로는 찾기 어렵습니다.

## 5. 규칙만으로 부족할 때

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | ✅ 완료 |
| 1 | 실습 파일 살펴보기 | ✅ 완료 |
| 2 | 검문소 없이 보내기 | ✅ 완료 |
| 3 | 검문소를 거쳐 보내기 (basic) | ✅ 완료 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | ✅ 완료 |
| 5 | 규칙만으로 부족할 때 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

규칙은 아는 형식만 찾습니다. 그래서 검문소에는 규칙 말고도 다음을 함께 둡니다.

- **보내는 데이터 자체를 줄입니다**: 요약에 주소가 꼭 필요한지부터 따집니다. 필요 없는 항목은 검문소 앞에서 아예 빼는 것이 가장 확실합니다(표 6-6 "보내는 데이터 최소화").
- **탐지 기술을 겹칩니다**: 실무 DLP와 AI Gateway의 민감정보 탐지는 규칙에 더해 주소·이름 같은 개체를 찾는 모델을 함께 씁니다. 그래도 완벽하지 않으므로, 확신이 낮은 문장은 보내지 않고 보류하는 정책을 둘 수 있습니다.
- **경계를 넘은 기록을 남깁니다**: 이 실습의 `received.log`처럼, 무엇이 나갔는지는 우리 쪽 검문소에서 기록해야 사고가 났을 때 영향 범위를 알 수 있습니다. 6-3에서 경고한 "디버깅 로그가 원문 그대로 외부 모니터링 서비스로 전송되는 경우"도 이 기록으로 찾아냅니다.

### 5.1. 보내기 전에 마지막으로 검사하기

"확신이 낮으면 보류"를 가장 단순하게 구현하면, 가명처리한 결과를 보내기 **직전에** 한 번 더 검사해 의심스러운 줄은 보내지 않는 것입니다. 아래 `hold_check()`는 주소처럼 보이는 줄(시·도와 "로·길"이 함께 있는 줄)을 찾아 보류합니다.

In [ ]:
import re

ADDRESS_HINT = re.compile(r"(서울|부산|대구|인천|광주|대전|울산|세종)\S*\s+\S+[구군]\s+\S+(로|길)")


def hold_check(text):
    """보낼 글을 줄마다 검사해 (보낼 줄, 보류할 줄)로 나눕니다."""
    send, hold = [], []
    for line in text.splitlines():
        (hold if ADDRESS_HINT.search(line) else send).append(line)
    return send, hold


send, hold = hold_check(RESULTS["strict 규칙"])
print(f"보낼 줄 {len(send)}개, 보류할 줄 {len(hold)}개")
for line in hold:
    print("⏸ 보류:", line)

보류된 줄은 사람이 확인하거나, 주소 부분을 지운 뒤 다시 보냅니다. 이 방식은 상담 3번 전체를 요약에서 빼므로 **업무 효용과 보호 사이의 균형**을 정해야 합니다. 과제 1에서 주소만 가리는 방법을 만들어 봅니다.

## 6. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | ✅ 완료 |
| 1 | 실습 파일 살펴보기 | ✅ 완료 |
| 2 | 검문소 없이 보내기 | ✅ 완료 |
| 3 | 검문소를 거쳐 보내기 (basic) | ✅ 완료 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | ✅ 완료 |
| 5 | 규칙만으로 부족할 때 | ✅ 완료 |
| 6 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 정리 | 예정 |

### 6.1. 체크포인트

- [ ] 검문소 없이 보냈을 때 외부로 나간 정보를 `received.log`에서 직접 확인했나요?
- [ ] 가명과 실명의 대응표가 어디에 남는지(사내 검문소 안) 설명할 수 있나요?
- [ ] 규칙을 강화한 뒤에도 경계를 넘은 정보는 무엇이었나요? 그 정보는 어떤 방법으로 막아야 할까요?

### 6.2. 과제 1: 주소 가리기 규칙 더하기

`gateway.pseudonymize()`의 결과에 주소 가리기를 더하는 `mask_address()`를 완성합니다. 주소 부분만 `주소X01`처럼 가명으로 바꾸고, 대응표에도 넣어 사내에서 복원할 수 있게 합니다. 셀을 실행하면 4.1과 같은 대조표에 "strict + 주소" 열이 생깁니다.

- 5.1의 `ADDRESS_HINT`를 출발점으로, 번지 숫자(`15`)까지 포함하도록 정규식을 넓혀 보세요.
- 주소를 가린 뒤에도 요약이 업무에 쓸 만한지("배송지를 회사로 바꾸고 싶다"는 내용이 남는지) 확인합니다.

In [ ]:
def mask_address(text, mapping):
    """주소를 가명으로 바꾸고 대응표에 더합니다."""
    # TODO: 주소를 찾는 정규식을 쓰고, 찾은 주소를 f"주소X{n:02d}"로 바꿉니다.
    return text, mapping


os.chdir(WORK)
masked, mapping = gateway.pseudonymize(original, gateway.load_customers(), "strict")
masked, mapping = mask_address(masked, mapping)
summary = gateway.call_external(masked)          # 외부로 보내기 (received.log에 남음)
os.chdir(LAB)
RESULTS["strict + 주소"] = masked
table(["정보", *RESULTS.keys()], [(k, *(cell(k, r) for r in RESULTS.values())) for k in SENSITIVE])
print("── 사내에서 복원한 요약")
print(gateway.restore(summary, mapping))

<details>
<summary>💡 힌트 보기</summary>

```python
ADDRESS = re.compile(r"(서울|부산|대구|인천|광주|대전|울산|세종)\S*\s+\S+[구군]\s+\S+(로|길)\s*\d+(-\d+)?")
n = 0
for found in [m.group(0) for m in ADDRESS.finditer(text)]:
    n += 1
    alias = f"주소X{n:02d}"
    text = text.replace(found, alias)
    mapping[alias] = found
```

이 정규식은 "시 구 로 번지" 형식만 찾습니다. "마포구 월드컵로 15"처럼 시를 빼고 말하거나, "회사 앞 편의점"처럼 말로 설명한 위치는 여전히 놓칩니다. 규칙은 계속 늘려야 하는 목록이라는 점, 그래서 **보내는 데이터 자체를 줄이는 것**이 먼저라는 점이 이 과제의 핵심입니다.
</details>

### 6.3. 과제 2: 검문소 기록의 보관 기준 정하기 (서술)

5단계에서 "무엇이 나갔는지 우리 쪽에 기록해야 한다"고 했습니다. 그런데 그 기록(`received.log`에 해당)에도 가명처리를 피해 간 개인정보가 들어 있을 수 있습니다. 아래 질문에 답해 보세요.

1. 검문소 기록은 누가 볼 수 있어야 하고, 얼마 동안 보관해야 할까요?
2. 기록에 원문을 그대로 남기지 않고도 "무엇이 나갔는지" 알 수 있는 방법은 무엇일까요? (예: 4.1처럼 항목별 개수만 남기기, 해시값 남기기)

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 7. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 흉내 AI 서비스 띄우기 | ✅ 완료 |
| 1 | 실습 파일 살펴보기 | ✅ 완료 |
| 2 | 검문소 없이 보내기 | ✅ 완료 |
| 3 | 검문소를 거쳐 보내기 (basic) | ✅ 완료 |
| 4 | 규칙을 강화해 다시 보내기 (strict) | ✅ 완료 |
| 5 | 규칙만으로 부족할 때 | ✅ 완료 |
| 6 | 체크포인트와 과제 | ✅ 완료 |
| 7 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [ ]:
fake_ai.terminate()
fake_ai.wait(timeout=5)
shutil.rmtree(WORK, ignore_errors=True)
sys.modules.pop("gateway", None)
print("흉내 서버 종료, 실습 폴더 삭제됨")

---
> 다음 실습 3에서는 이 검문소가 들어갈 자리인 AI Gateway를 직접 띄우고, 공급자를 바꾸는 Exit 리허설을 합니다. 두 공급자 모두에서 남는 위험은 이 실습처럼 우리 쪽 Gateway에서 막아야 합니다.